# N131 · 概率基础与随机采样

**目标：** 用等概率推导而非直方图宣称无偏。

**先修：** N042, N031, N124。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 概率空间；条件概率；期望；Fisher–Yates；蓄水池；加权采样；拒绝采样。

## 从问题到算法

“看起来随机”不等于每个结果等概率。Fisher–Yates每次从尚未固定的位置中均匀选一个；蓄水池在第i项出现时以k/i概率保留它；加权抽样把整数权重变成等长基本抽签区间。rand10通过拒绝不能均分为10份的尾部样本消除偏差。

## 最小实现

**本章接口：** `fisher_yates(a, rng)`、`reservoir_sample(stream, k, rng)`、`WeightedSampler`、`rand10(rand7)`

In [1]:
from bisect import bisect_right
from random import Random

def fisher_yates(a,rng):
    for i in range(len(a)-1,0,-1):
        j=rng.randrange(i+1); a[i],a[j]=a[j],a[i]
    return a

def reservoir_sample(stream,k,rng):
    if k<0: raise ValueError('nonnegative reservoir size required')
    reservoir=[]
    for i,x in enumerate(stream):
        if i<k: reservoir.append(x)
        elif k:
            j=rng.randrange(i+1)
            if j<k: reservoir[j]=x
    return reservoir

class WeightedSampler:
    def __init__(self,weights,rng=None):
        if any(not isinstance(w,int) or w<0 for w in weights) or sum(weights)<=0: raise ValueError('nonnegative integer weights with positive total required')
        self.prefix=[]; total=0
        for w in weights: total+=w; self.prefix.append(total)
        self.total=total; self.rng=Random() if rng is None else rng
    def pick_index(self):
        ticket=self.rng.randrange(self.total)
        return bisect_right(self.prefix,ticket)

def rand10(rand7):
    while True:
        a,b=rand7(),rand7()
        if not 1<=a<=7 or not 1<=b<=7: raise ValueError('rand7 must return 1..7')
        ticket=(a-1)*7+(b-1)
        if ticket<40: return ticket%10+1

## 正确性、前提与复杂度

洗牌每个最终排列对应唯一抽签路径，概率为1/n!。蓄水池中旧元素在第i步后存活概率(k/(i−1))·(1−1/i)=k/i，新元素被保留概率也是k/i；完整均匀k子集性质可继续按子集归纳。加权区间包含恰好w[i]张票；拒绝抽样接受的40张票对10个结果各贡献4张。

**代价：** 洗牌O(n)时间O(1)辅助；蓄水池O(n)时间O(k)空间且仅遍历一次；加权初始化O(n)、每次O(log n)；rand10期望调用rand7次数2·49/40，无确定有限最坏次数。统计频率仅辅助诊断，测试主要使用精确轨迹枚举。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

from itertools import product
from collections import Counter
class FixedRng:
    def __init__(self,values): self.values=iter(values)
    def randrange(self,stop):
        value=next(self.values); assert 0<=value<stop; return value
counts=Counter(tuple(fisher_yates([0,1,2],FixedRng([a,b]))) for a,b in product(range(3),range(2)))
show_table(['排列','六条等概率抽签轨迹中的次数'],sorted(counts.items()))

排列,六条等概率抽签轨迹中的次数
"(0, 1, 2)",1
"(0, 2, 1)",1
"(1, 0, 2)",1
"(1, 2, 0)",1
"(2, 0, 1)",1
"(2, 1, 0)",1


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
from itertools import product
from collections import Counter
class FixedRng:
    def __init__(self,values): self.values=iter(values)
    def randrange(self,stop):
        value=next(self.values); assert 0<=value<stop; return value
counts=Counter(tuple(fisher_yates([0,1,2],FixedRng([a,b]))) for a,b in product(range(3),range(2)))
assert len(counts)==6 and set(counts.values())=={1}
subsets=Counter(tuple(sorted(reservoir_sample(range(4),2,FixedRng([a,b])))) for a,b in product(range(3),range(4)))
assert len(subsets)==6 and set(subsets.values())=={2}
assert [WeightedSampler([0,2,0,3],FixedRng([i])).pick_index() for i in range(5)]==[1,1,3,3,3]
assert reservoir_sample(range(10),3,Random(131))==reservoir_sample(range(10),3,Random(131))
assert reservoir_sample(iter([1,2]),5,Random(1))==[1,2]
assert reservoir_sample(range(5),0,Random(1))==[]
accepted=Counter()
for a,b in product(range(1,8),repeat=2):
    if (a-1)*7+b-1<40:
        values=iter([a,b]); accepted[rand10(lambda:next(values))]+=1
assert accepted==Counter({i:4 for i in range(1,11)})
values=iter([7,7,1,1]); assert rand10(lambda:next(values))==1
print('N131: 所有本章断言通过')

N131: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 证明单元素蓄水池每项概率1/n。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 构造错误全范围交换shuffle的偏差。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 384. Shuffle an Array（canonical）
- 398. Random Pick Index（canonical）
- 528. Random Pick with Weight（canonical）
- 470. Implement Rand10() Using Rand7()（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。